# PN Junction Diode

## Understanding the Fundamental Semiconductor Device

The PN junction diode is the most fundamental semiconductor device and forms the basis for understanding more complex devices like transistors and solar cells.

**Learning Objectives:**
- Understand PN junction physics
- Simulate equilibrium band diagrams with PADRE
- Analyze forward and reverse bias characteristics
- Extract diode parameters from simulated I-V curves

In [ ]:
# Setup: Load PADRE environment (required on nanoHUB)
# This cell loads the PADRE simulator into your environment.
# If running locally with PADRE already in your PATH, this will be skipped gracefully.

from nanohubpadre import use

# Load the PADRE simulator environment
%use padre-2.4E-r15

print("PADRE environment setup complete.")

---

## Device Parameters Reference

The `describe()` function shows all available parameters for the PN diode factory, including geometry, doping, physical models, and sweep options.

In [ ]:
from nanohubpadre import Simulation

# Show all available parameters for the PN diode
Simulation.describe('pn_diode')

---

## 1. PN Junction Physics

### 1.1 Structure

A PN junction consists of:
- **P-region**: Semiconductor doped with acceptors (holes are majority carriers)
- **N-region**: Semiconductor doped with donors (electrons are majority carriers)
- **Depletion region**: The space charge region at the junction

```
    P-region          |    N-region
  (Acceptors Na)      |  (Donors Nd)
                      |
  Holes (majority)    |  Electrons (majority)
  Electrons (minority)|  Holes (minority)
                      |
         <-- Depletion Region -->
```

### 1.2 Key Parameters

- **Built-in potential** ($V_{bi}$): The potential barrier at equilibrium
  $$V_{bi} = \frac{kT}{q} \ln\left(\frac{N_a N_d}{n_i^2}\right)$$

- **Depletion width** ($W$): Width of the space charge region
  $$W = \sqrt{\frac{2\epsilon_s}{q}\left(\frac{1}{N_a} + \frac{1}{N_d}\right)(V_{bi} - V)}$$

- **Diode current** (ideal):
  $$I = I_s\left(e^{qV/kT} - 1\right)$$

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from nanohubpadre import create_pn_diode

# Physical constants
q = 1.6e-19      # Elementary charge (C)
kT = 0.0259      # Thermal voltage at 300K (eV)
ni = 1.5e10      # Intrinsic carrier concentration for Si (cm^-3)
eps_si = 11.7 * 8.85e-14  # Silicon permittivity (F/cm)

# Calculate built-in potential
Na = 1e17  # Acceptor concentration (cm^-3)
Nd = 1e17  # Donor concentration (cm^-3)

Vbi = kT * np.log(Na * Nd / ni**2)
print(f"Built-in potential Vbi = {Vbi:.3f} V")

# Calculate depletion width at equilibrium
W = np.sqrt(2 * eps_si / q * (1/Na + 1/Nd) * Vbi) * 1e4  # Convert to microns
print(f"Depletion width W = {W:.4f} um")

---

## 2. Creating and Running a PN Diode Simulation

Let's create a basic PN diode and run PADRE to examine its equilibrium properties.

In [ ]:
# Create a PN diode with symmetric doping
sim_eq = create_pn_diode(
    # Geometry
    length=2.0,              # 2 um total length
    width=1.0,               # 1 um width
    junction_position=0.5,   # Junction at center
    
    # Mesh
    nx=200,                  # Fine mesh for accurate results
    ny=3,                    # Minimal y-mesh (quasi-1D)
    
    # Doping
    p_doping=1e17,           # P-region: 1e17 cm^-3
    n_doping=1e17,           # N-region: 1e17 cm^-3
    
    # Models
    temperature=300,         # Room temperature
    srh=True,               # SRH recombination
    conmob=True,            # Concentration-dependent mobility
    
    # Output - log band diagram at equilibrium
    log_bands_eq=True
)

print("PN Diode Simulation Created")
print("="*40)
print(f"Device length: 2.0 um")
print(f"P-doping: 1e17 cm^-3")
print(f"N-doping: 1e17 cm^-3")
print(f"Junction position: 1.0 um (center)")

In [ ]:
# Visualize the device structure
sim_eq.device_schematic()

In [ ]:
# View the generated PADRE input deck
print("Generated PADRE Input Deck:")
print("="*60)
print(sim_eq.generate_deck())

In [ ]:
# Run the PADRE simulation
print("Running PADRE simulation...")
result = sim_eq.run()

if result.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result.stderr}")


---

## 3. Equilibrium Band Diagram

At equilibrium (zero bias), the Fermi level is constant throughout the device. The band bending creates the built-in potential barrier.

Let's plot the simulated band diagram from PADRE.

In [ ]:
# Plot the equilibrium band diagram from PADRE simulation
print("\nAvailable outputs:")
print(sim_eq.outputs.summary())

In [ ]:
# Plot band diagram using the built-in plotting method
sim_eq.plot_band_diagram(title="PN Junction Equilibrium Band Diagram (PADRE Simulation)", backend="plotly");


In [ ]:
# Get the raw band data for custom plotting
ec_data = sim_eq.outputs.get("cbeq")  # Conduction band
ev_data = sim_eq.outputs.get("vbeq")  # Valence band

if ec_data is not None and ev_data is not None:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=ec_data.x, y=ec_data.y, mode='lines',
        name='Ec (Conduction Band)',
        line=dict(color='blue', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=ev_data.x, y=ev_data.y, mode='lines',
        name='Ev (Valence Band)',
        line=dict(color='red', width=2)
    ))

    # Fermi level (constant at equilibrium — use midgap as reference)
    Ef = (ec_data.y[0] + ev_data.y[0]) / 2
    fig.add_hline(y=0, line_dash="dash", line_color="green", line_width=1.5,
                  annotation_text="Ef (Fermi Level)", annotation_position="top right")

    # Junction marker
    junction = 1.0  # um
    fig.add_vline(x=junction, line_dash="dot", line_color="gray", line_width=1,
                  annotation_text="Junction", annotation_position="top")

    # Region labels via annotations
    fig.add_annotation(x=0.3, y=max(ec_data.y) * 0.9, text="P-region",
                       showarrow=False, font=dict(size=14))
    fig.add_annotation(x=1.5, y=max(ec_data.y) * 0.9, text="N-region",
                       showarrow=False, font=dict(size=14))

    fig.update_layout(
        title="PN Junction Equilibrium Band Diagram (PADRE Simulation)",
        xaxis_title="Position (μm)",
        yaxis_title="Energy (eV)",
        width=800, height=500,
        template="plotly_white",
        legend=dict(x=0.02, y=0.98)
    )
    fig.show()

    # Calculate built-in potential from simulation
    Vbi_sim = ec_data.y[0] - ec_data.y[-1]
    print(f"\nSimulated built-in potential: Vbi = {abs(Vbi_sim):.3f} V")
    print(f"Theoretical built-in potential: Vbi = {Vbi:.3f} V")
else:
    print("Band data not available. Check simulation output.")

### Understanding the Band Diagram

The equilibrium band diagram shows:

1. **P-region (left)**: 
   - Fermi level near valence band
   - Bands curved upward toward junction

2. **N-region (right)**:
   - Fermi level near conduction band
   - Bands curved downward toward junction

3. **Junction**:
   - Band bending = built-in potential
   - Depletion region where bands curve

---

## 4. Forward Bias I-V Characteristics

Under forward bias (positive voltage on P-side):
- Barrier height decreases
- Current increases exponentially
- Minority carrier injection increases

Let's simulate the forward bias I-V curve with PADRE.

In [ ]:
# Create diode simulation with forward bias sweep
sim_forward = create_pn_diode(
    length=2.0,
    junction_position=0.5,
    p_doping=1e17,
    n_doping=1e17,
    nx=200,
    ny=3,
    
    # Models
    temperature=300,
    srh=True,
    conmob=True,
    
    # Enable I-V logging
    log_iv=True,
    iv_file="forward_iv",
    
    # Forward bias sweep: 0 to 0.8V in 0.02V steps
    forward_sweep=(0.0, 0.8, 0.02),
    
    # Also log band diagrams at final bias
    log_bands_eq=True
)

print("Forward Bias Simulation Configured")
print("="*40)
print("Voltage sweep: 0V to 0.8V")
print("Step size: 0.02V")
print("Number of bias points:", int(0.8/0.02) + 1)

In [ ]:
# Run the forward bias simulation
print("Running forward bias simulation...")
result_fwd = sim_forward.run()

if result_fwd.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result_fwd.stderr}")


In [ ]:
# Plot the simulated I-V characteristic
sim_forward.plot_iv(title="Forward I-V Characteristic")


In [ ]:
# Extract ideality factor from the simulation data
import numpy as np

kT = 0.025852  # thermal voltage at 300 K (V)

# Load the I-V log that the forward-bias run just wrote.
iv = sim_forward.get_iv_data()

# Electrode 1 is the P side (anode); it is the electrode being swept.
V_sim = iv.get_voltages(1)
I_sim = np.abs(iv.get_currents(1))

# Terminal-current continuity (I1 == -I2) tells real physics from round-off.
# Points that fail it sit at the solver's noise floor and must not be fitted.
suspect = iv.check_continuity(tolerance=0.01, warn=False)
print(f"{suspect.sum()} of {len(suspect)} bias points are at the noise floor "
      f"and are excluded from the fit.")

mask = (V_sim > 0.3) & (V_sim < 0.6) & (I_sim > 0) & (~suspect)

if mask.sum() > 2:
    coeffs = np.polyfit(V_sim[mask], np.log(I_sim[mask]), 1)
    n = 1 / (kT * coeffs[0])          # slope = q/(n·kT) in eV units
    Is_sim = np.exp(coeffs[1])

    print("\nExtracted Diode Parameters")
    print("=" * 45)
    print(f"  Fit window                            : {mask.sum()} points, "
          f"{V_sim[mask].min():.2f}-{V_sim[mask].max():.2f} V")
    print(f"  Ideality factor  n  (sim, linear fit) : {n:.2f}")
    print(f"  Saturation current Is (sim, fit)      : {Is_sim:.3e} A")
    print()
    print("  n > 1 indicates recombination in the depletion region")
    print("  (ideal n = 1 is pure diffusion; n = 2 is pure recombination).")
    print()
    print("  Note: with the default 1 us lifetimes this device is short-base -")
    print("  the current is set by the ohmic contacts, not by bulk")
    print("  recombination, so n comes out near 1.0. Re-run with")
    print("  taun0=taup0=1e-9 to see the depletion-recombination regime.")
else:
    print("Not enough resolved bias points in 0.3-0.6 V to fit.")


---

## 5. Reverse Bias Characteristics

Under reverse bias (negative voltage on P-side):
- Barrier height increases
- Depletion region widens
- Only small reverse saturation current flows

In [ ]:
# Create diode with reverse bias sweep
sim_reverse = create_pn_diode(
    length=2.0,
    junction_position=0.5,
    p_doping=1e17,
    n_doping=1e17,
    nx=200,
    ny=3,
    
    temperature=300,
    srh=True,
    
    # Enable I-V logging
    log_iv=True,
    iv_file="reverse_iv",
    
    # Reverse bias sweep: 0 to -5V
    reverse_sweep=(0.0, -5.0, -0.25)
)

print("Reverse Bias Simulation Configured")
print("="*40)
print("Voltage sweep: 0V to -5V")
print("Step size: -0.25V")

In [ ]:
# Run the reverse bias simulation
print("Running reverse bias simulation...")
result_rev = sim_reverse.run()

if result_rev.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result_rev.stderr}")


In [ ]:
# Plot reverse bias characteristics — linear and log scale
try:
    iv_rev = sim_reverse.get_iv_data()
    V_rev, I_rev = iv_rev.get_iv_data(electrode=2)
    V_rev = np.array(V_rev)
    I_rev = np.abs(np.array(I_rev))

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "Reverse I-V (Linear Scale)",
        "Reverse I-V (Log Scale)"
    ))

    # ── Linear panel ──
    fig.add_trace(go.Scatter(
        x=np.abs(V_rev), y=I_rev * 1e12, mode='lines+markers',
        name='PADRE Simulation',
        line=dict(color='red', width=2),
        marker=dict(size=4)
    ), row=1, col=1)

    # ── Log panel ──
    mask = I_rev > 0
    fig.add_trace(go.Scatter(
        x=np.abs(V_rev[mask]), y=I_rev[mask], mode='lines+markers',
        name='PADRE Simulation',
        line=dict(color='red', width=2),
        marker=dict(size=4),
        showlegend=False
    ), row=1, col=2)

    fig.update_layout(
        width=1000, height=450,
        template="plotly_white",
    )
    fig.update_xaxes(title_text="Reverse Bias (V)")
    fig.update_yaxes(title_text="Reverse Current (pA)", row=1, col=1)
    fig.update_yaxes(title_text="Reverse Current (A)", type="log", row=1, col=2)
    fig.show()

    print()
    print(f"Reverse saturation current: Is ≈ {np.min(I_rev):.3e} A")
    W_5V = np.sqrt(2 * eps_si / q * (1/Na + 1/Nd) * (Vbi + 5.0)) * 1e4
    print(f"Depletion width at 5 V reverse bias: W = {W_5V:.4f} µm")

except Exception as e:
    print(f"Could not plot reverse I-V: {e}")

---

## 6. Effect of Doping Concentration

Let's explore how doping affects the diode characteristics using PADRE simulations.

In [ ]:
# Simulate diodes with different doping levels
doping_levels = [1e16, 1e17, 1e18]
results = {}

print("Simulating diodes with different doping levels:")
print("="*50)

for doping in doping_levels:
    print(f"\nDoping: {doping:.0e} cm^-3")
    
    sim = create_pn_diode(
        length=2.0,
        junction_position=0.5,
        p_doping=doping,
        n_doping=doping,
        nx=150,
        ny=3,
        temperature=300,
        srh=True,
        log_iv=True,
        iv_file=f"iv_{doping:.0e}",
        forward_sweep=(0.0, 0.8, 0.02)
    )
    
    result = sim.run()
    
    if result.returncode != 0:
        raise RuntimeError(f"Simulation failed:\n{result.stderr}")
    # Calculate theoretical Vbi
    Vbi_calc = kT * np.log(doping * doping / ni**2)
    print(f"  Vbi (theoretical) = {Vbi_calc:.3f} V")
    
    try:
        iv_data = sim.get_iv_data()
        V, I = iv_data.get_iv_data(electrode=2)
        results[doping] = (np.array(V), np.abs(np.array(I)))
        print(f"  Simulation successful")
    except:
        print(f"  Could not parse I-V data")


In [ ]:
# Plot comparison of different doping levels
if results:
    colors = ['blue', 'green', 'red']

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "I-V Characteristics (Linear)",
        "I-V Characteristics (Log Scale)"
    ))

    for (doping, (V, I)), color in zip(results.items(), colors):
        label = f'Na=Nd={doping:.0e} cm\u207b\u00b3'

        # ── Linear panel ──
        fig.add_trace(go.Scatter(
            x=V, y=I * 1e6, mode='lines',
            name=label,
            line=dict(color=color, width=2)
        ), row=1, col=1)

        # ── Log panel ──
        fig.add_trace(go.Scatter(
            x=V, y=I, mode='lines',
            name=label,
            line=dict(color=color, width=2),
            showlegend=False
        ), row=1, col=2)

    fig.update_layout(
        width=1000, height=450,
        template="plotly_white",
    )
    fig.update_xaxes(title_text="Voltage (V)")
    fig.update_yaxes(title_text="Current (µA)", row=1, col=1)
    fig.update_yaxes(title_text="Current (A)", type="log", row=1, col=2)
    fig.show()

---

## 7. Complete Simulation with Band Diagrams Under Bias

Let's create a complete simulation that shows band diagrams at different bias conditions.

In [ ]:
# Complete PN diode characterization with band diagrams
sim_complete = create_pn_diode(
    # Device geometry
    length=2.0,
    width=1.0,
    junction_position=0.5,
    
    # High-resolution mesh
    nx=200,
    ny=3,
    
    # Doping
    p_doping=1e17,
    n_doping=1e17,
    
    # Physical models
    temperature=300,
    srh=True,
    conmob=True,
    fldmob=True,
    
    # Material parameters
    taun0=1e-6,  # Electron lifetime
    taup0=1e-6,  # Hole lifetime
    
    # Output logging
    log_iv=True,
    iv_file="complete_iv",
    log_bands_eq=True,
    log_bands_bias=True,
    
    # Forward bias sweep
    forward_sweep=(0.0, 0.7, 0.1)
)

print("Complete PN Diode Simulation")
print("="*50)
print("Running simulation...")

result_complete = sim_complete.run()

if result_complete.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result_complete.stderr}")
print("\nOutputs generated:")
print(sim_complete.outputs.summary())


In [ ]:
# Plot all band diagram sets (equilibrium and biased)
try:
    sim_complete.plot_band_diagram(title="Band Diagrams at Different Bias Conditions", backend="plotly");
except Exception as e:
    print(f"Could not plot band diagrams: {e}")

---

## 8. Recombination, Electric Field, and Excess Carriers Under Bias

Three quantities reveal the internal physics of the diode as bias changes:

- **Recombination rate** ($R$): At equilibrium generation balances recombination everywhere. Under forward bias, excess carriers are injected into the quasi-neutral regions and recombine there — the rate peaks near the junction and in the bulk.

- **Electric field** ($E$): At equilibrium the field exists only inside the depletion region. Forward bias shrinks the depletion region and reduces the peak field; reverse bias does the opposite.

- **Excess carrier density**: The minority-carrier concentrations (electrons in P, holes in N) rise exponentially with forward bias. Plotting $n$, $p$, and the net carrier density $(n - p)$ together shows where injection dominates and where the device is in quasi-neutrality.

We build a single simulation that solves equilibrium, then steps through several forward bias points. At each step we log all three quantities along the same 1-D cut through the device.

In [ ]:
# ── Physics-profile simulation using the device factory ──
# log_physics_at triggers step-by-step bias ramp with
# recomb / e_field / electrons / holes / net_carrier logged at each point.

BIAS_POINTS = [0.0, 0.2, 0.4, 0.6]   # V  (forward bias on electrode 2)
x_junc = 1.0                           # junction position in µm

sim_phys = create_pn_diode(
    length=2.0,
    width=1.0,
    junction_position=0.5,
    nx=200,
    ny=3,
    p_doping=1e17,
    n_doping=1e17,
    temperature=300,
    srh=True,
    conmob=True,
    fldmob=True,
    taun0=1e-6,
    taup0=1e-6,
    log_iv=True,
    iv_file="phys_iv",
    log_physics_at=BIAS_POINTS,
)

print("Simulation deck ready.")
print(f"Bias points logged: {BIAS_POINTS}")
print(f"Expected output files: {len(BIAS_POINTS) * 5} profile files + IV log")


In [ ]:
# Run the physics-profiles simulation
print("Running physics-profiles simulation...")
result_phys = sim_phys.run()

if result_phys.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result_phys.stderr}")


### 8.1 Recombination Rate

At equilibrium thermal generation and recombination balance ($G = R$) everywhere.
Under forward bias minority carriers are injected and recombination rises sharply
in the quasi-neutral regions on either side of the junction.

In [ ]:
# ── Recombination rate at each bias point ──
LABELS = {"eq": "0.0 V (equilibrium)", "v0p2": "0.2 V", "v0p4": "0.4 V", "v0p6": "0.6 V"}
COLORS = ["gray", "blue", "green", "red"]

fig = go.Figure()
for tag, color in zip(["eq", "v0p2", "v0p4", "v0p6"], COLORS):
    data = sim_phys.outputs.get(f"recomb_{tag}")
    if data is not None and len(data.x) > 0:
        fig.add_trace(go.Scatter(
            x=data.x, y=np.abs(data.y), mode="lines",
            name=LABELS[tag],
            line=dict(color=color, width=2)
        ))

fig.update_layout(
    title="Recombination Rate vs Position",
    xaxis_title="Position (μm)",
    yaxis_title="Recombination Rate (/cm³·s)",
    yaxis_type="log",
    width=750, height=500,
    template="plotly_white"
)
# junction marker
fig.add_vline(x=x_junc, line_dash="dot", line_color="gray", line_width=1,
              annotation_text="Junction", annotation_position="top")
fig.show()

### 8.2 Electric Field

The built-in electric field is confined to the depletion region.
Forward bias narrows the depletion region and lowers the peak field;
the field goes to zero in the quasi-neutral bulk on both sides.

In [ ]:
# ── Electric field at each bias point ──
fig = go.Figure()
for tag, color in zip(["eq", "v0p2", "v0p4", "v0p6"], COLORS):
    data = sim_phys.outputs.get(f"efield_{tag}")
    if data is not None and len(data.x) > 0:
        fig.add_trace(go.Scatter(
            x=data.x, y=data.y, mode="lines",
            name=LABELS[tag],
            line=dict(color=color, width=2)
        ))

fig.update_layout(
    title="Electric Field vs Position",
    xaxis_title="Position (µm)",
    yaxis_title="Electric Field (V/cm)",
    width=750, height=500,
    template="plotly_white"
)
fig.add_vline(x=x_junc, line_dash="dot", line_color="gray", line_width=1,
              annotation_text="Junction", annotation_position="top")
fig.show()

### 8.3 Carrier Concentrations and Excess Carriers

**Top panel** — electron ($n$) and hole ($p$) concentrations at the highest
forward bias (0.6 V).  The minority-carrier tails (electrons in P, holes in N)
are the injected excess carriers that recombine in the bulk.

**Bottom panel** — the net carrier density $n - p$ at every logged bias point.
At equilibrium $n - p$ is antisymmetric about the junction and very small in
the bulk.  Under forward bias the injected minorities push $n - p$ positive
deep into the P-side and negative deep into the N-side.

In [ ]:
# ── Top: n and p at 0.6 V (log scale) ──
fig_top = go.Figure()

n_06 = sim_phys.outputs.get("n_v0p6")
p_06 = sim_phys.outputs.get("p_v0p6")
if n_06 is not None:
    fig_top.add_trace(go.Scatter(
        x=n_06.x, y=n_06.y, mode="lines",
        name="Electrons n", line=dict(color="blue", width=2)
    ))
if p_06 is not None:
    fig_top.add_trace(go.Scatter(
        x=p_06.x, y=p_06.y, mode="lines",
        name="Holes p", line=dict(color="red", width=2)
    ))

fig_top.update_layout(
    title="Carrier Concentrations at 0.6 V Forward Bias",
    xaxis_title="Position (µm)",
    yaxis_title="Concentration (/cm³)",
    yaxis_type="log",
    width=750, height=450,
    template="plotly_white"
)
fig_top.add_vline(x=x_junc, line_dash="dot", line_color="gray", line_width=1,
                  annotation_text="Junction", annotation_position="top")
fig_top.show()

# ── Bottom: net carrier (n - p) at every bias (linear scale) ──
fig_bot = go.Figure()
for tag, color in zip(["eq", "v0p2", "v0p4", "v0p6"], COLORS):
    data = sim_phys.outputs.get(f"netcar_{tag}")
    if data is not None and len(data.x) > 0:
        fig_bot.add_trace(go.Scatter(
            x=data.x, y=data.y, mode="lines",
            name=LABELS[tag],
            line=dict(color=color, width=2)
        ))

fig_bot.update_layout(
    title="Net Carrier Density (n − p) vs Position",
    xaxis_title="Position (µm)",
    yaxis_title="n − p  (/cm³)",
    width=750, height=450,
    template="plotly_white"
)
fig_bot.add_vline(x=x_junc, line_dash="dot", line_color="gray", line_width=1,
                  annotation_text="Junction", annotation_position="top")
fig_bot.show()

---

## 9. Exercises

### Exercise 1: Asymmetric Junction
Create a diode with different doping on each side (Na = 1e16, Nd = 1e18). How does this affect:
- The built-in potential?
- The depletion region distribution?
- The I-V characteristics?

In [ ]:
# Exercise 1: Asymmetric junction simulation
sim_asymmetric = create_pn_diode(
    length=2.0,
    junction_position=0.5,
    p_doping=1e16,   # Lightly doped P-side
    n_doping=1e18,   # Heavily doped N-side
    nx=200,
    ny=3,
    log_bands_eq=True,
    log_iv=True,
    forward_sweep=(0.0, 0.8, 0.02)
)

print("Running asymmetric junction simulation...")
result_asym = sim_asymmetric.run()

if result_asym.returncode != 0:
    raise RuntimeError(f"Simulation failed:\n{result_asym.stderr}")
# Calculate theoretical values
Na_asym = 1e16
Nd_asym = 1e18
Vbi_asym = kT * np.log(Na_asym * Nd_asym / ni**2)

# Depletion widths on each side
xp = np.sqrt(2 * eps_si * Vbi_asym / q * Nd_asym / (Na_asym * (Na_asym + Nd_asym))) * 1e4
xn = np.sqrt(2 * eps_si * Vbi_asym / q * Na_asym / (Nd_asym * (Na_asym + Nd_asym))) * 1e4

print("\nAsymmetric Junction Analysis")
print("=" * 40)
print(f"Na = {Na_asym:.0e} cm^-3")
print(f"Nd = {Nd_asym:.0e} cm^-3")
print(f"Vbi = {Vbi_asym:.3f} V")
print(f"Depletion in P-region: {xp:.4f} μm")
print(f"Depletion in N-region: {xn:.4f} μm")
print(f"Total depletion width: {xp + xn:.4f} μm")
print(f"\nNote: Most depletion is in the lightly-doped P-region")

# Plot band diagram
sim_asymmetric.plot_band_diagram(title="Asymmetric PN Junction Band Diagram", backend="plotly");


### Exercise 2: Temperature Dependence
Simulate the diode at different temperatures. How does the I-V curve change?

In [ ]:
# Exercise 2: Temperature dependence
temperatures = [300, 350, 400]
temp_results = {}

print("Simulating at different temperatures:")
print("=" * 50)

for T in temperatures:
    print(f"\nTemperature: {T} K")

    sim_T = create_pn_diode(
        length=2.0,
        junction_position=0.5,
        p_doping=1e17,
        n_doping=1e17,
        nx=150,
        ny=3,
        temperature=T,
        srh=True,
        log_iv=True,
        forward_sweep=(0.0, 0.8, 0.02)
    )

    result_T = sim_T.run()

    if result_T.returncode != 0:
        raise RuntimeError(f"Simulation failed:\n{result_T.stderr}")
    try:
        iv_data = sim_T.get_iv_data()
        V, I = iv_data.get_iv_data(electrode=2)
        temp_results[T] = (np.array(V), np.abs(np.array(I)))
        print(f"  Thermal voltage kT/q = {T * 8.617e-5:.4f} V")
    except:
        print(f"  Could not parse data")

# Plot temperature comparison
if temp_results:
    fig = go.Figure()

    for T, (V, I) in temp_results.items():
        fig.add_trace(go.Scatter(
            x=V, y=I, mode='lines',
            name=f'T = {T} K',
            line=dict(width=2)
        ))

    fig.update_layout(
        title="Temperature Effect on PN Diode I-V",
        xaxis_title="Voltage (V)",
        yaxis_title="Current (A)",
        yaxis_type="log",
        width=700, height=500,
        template="plotly_white",
    )
    fig.show()

    print("\nKey observations:")
    print("- Higher temperature → Higher saturation current")
    print("- Turn-on voltage decreases with temperature")
    print("- Slope (q/nkT) decreases with temperature")

---

## Summary

In this notebook, you learned:

1. **PN Junction Physics**: Built-in potential, depletion region, I-V characteristics
2. **Running PADRE Simulations**: Using `create_pn_diode()` and `sim.run()`
3. **Band Diagrams**: Extracting and plotting band structure from PADRE output
4. **I-V Characteristics**: Forward and reverse bias behavior from simulation
5. **Parameter Extraction**: Ideality factor, saturation current from simulated data
6. **Parameter Effects**: How doping and temperature affect device behavior
7. **Complete Simulations**: Multi-step bias sweeps with band-diagram logging
8. **Internal Physics Profiles**: How recombination rate, electric field, and carrier densities evolve with applied bias — using low-level `Plot1D` commands to capture snapshots at discrete bias points

**Key Equations:**
- Built-in potential: $V_{bi} = \frac{kT}{q}\ln\left(\frac{N_aN_d}{n_i^2}\right)$
- Ideal diode: $I = I_s(e^{qV/kT} - 1)$
- SRH recombination: $R = \frac{np - n_i^2}{\tau_p(n + n_1) + \tau_n(p + p_1)}$

**Next**: [03 - Schottky Diode](03_Schottky_Diode.ipynb) - Metal-semiconductor junctions